# 🏠 End-to-End Regression Pipeline with `model_track`

This notebook demonstrates the **complete regression modeling workflow** using the `model_track` library — from raw data to model evaluation, stability monitoring, and context persistence.

We use the **California Housing dataset** (scikit-learn): 8 numeric features, continuous target (`median_house_value`), ~20k samples.

## Pipeline Overview

```
Raw Data
  └─► 1. Data Setup & Train/OOT Split
        └─► 2. Data Auditing (DataAuditor, TypeDetector)
              └─► 3. Feature Selection (RegressionSelector)
                    └─► 4. Model Training (LightGBM Regressor)
                          └─► 5. Evaluation (RegressionEvaluator)
                                └─► 6. Stability Monitoring (RegressionPSI, StabilityReport)
                                      └─► 7. Context Persistence (ProjectContext)
```

## 📦 Imports

In [ ]:
from datetime import datetime

import lightgbm as lgb
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

# model_track — public API only
from model_track import ProjectContext, TaskType
from model_track.evaluation import RegressionEvaluator
from model_track.preprocessing import DataAuditor, TypeDetector
from model_track.stability import PSICalculator, RegressionPSI, StabilityReport
from model_track.stats import RegressionSelector

print("All imports OK ✅")

---
## 1. 📂 Data Setup & Train/OOT Split

We load the California Housing dataset and split into:
- **Development set** (70%): used for fitting all transformers and the model.
- **OOT (Out-of-Time) set** (30%): simulates unseen production data for evaluation and stability checks.

The target is `median_house_value` (in $100k units) — a continuous regression target.

In [ ]:
# Load dataset
housing = fetch_california_housing(as_frame=True)
df = housing.frame.copy()
df = df.rename(columns={"MedHouseVal": "target"})

TARGET = "target"
FEATURES = [c for c in df.columns if c != TARGET]

print(f"Dataset shape : {df.shape}")
print(f"Target        : {TARGET}")
print(f"Features      : {FEATURES}")
print("\nTarget stats:")
print(df[TARGET].describe().to_string())
df.head()

In [ ]:
# 70% dev / 30% OOT split (random, no stratify for regression)
df_dev, df_oot = train_test_split(df, test_size=0.30, random_state=42)
df_dev = df_dev.reset_index(drop=True)
df_oot = df_oot.reset_index(drop=True)

print(f"Dev set : {len(df_dev):6,} rows")
print(f"OOT set : {len(df_oot):6,} rows")

---
## 2. 🔍 Data Auditing

`DataAuditor` provides a statistical summary — missing values, types, cardinality. `TypeDetector` classifies each feature, informing downstream selection.

In [ ]:
# Statistical summary
auditor = DataAuditor()
summary = auditor.get_summary(df_dev)
print("📊 Data Audit Summary:")
summary

In [ ]:
# Detect feature types
detector = TypeDetector(target=TARGET)
feature_types = detector.detect(df_dev)

numerical_features = feature_types.get("numerical", [])
categorical_features = feature_types.get("categorical_low", []) + feature_types.get(
    "categorical_high", []
)

print(f"Numerical   : {len(numerical_features)} → {numerical_features}")
print(f"Categorical : {len(categorical_features)} → {categorical_features}")

---
## 3. 🎯 Feature Selection with `RegressionSelector`

`RegressionSelector` selects features using two complementary filters:

1. **Correlation filter**: keeps only features with `|Spearman correlation|` ≥ `min_correlation` with the target (default 0.05).
2. **Multicollinearity filter**: computes **VIF (Variance Inflation Factor)** iteratively — features with VIF > `vif_threshold` (default 10.0) are dropped to reduce redundancy.

Both Pearson and Spearman methods are supported; Spearman is recommended for non-linear relationships.

In [ ]:
selector = RegressionSelector(
    method="spearman",
    min_correlation=0.05,
    correlation_threshold=0.9,
    vif_threshold=10.0,
)
selector.fit(df_dev, target=TARGET, features=numerical_features)

SELECTED_FEATURES = selector.selected_features_
DROPPED_FEATURES = selector.dropped_features_

print(f"✅ Selected : {len(SELECTED_FEATURES)} features → {SELECTED_FEATURES}")
print(f"❌ Dropped  : {len(DROPPED_FEATURES)} features → {DROPPED_FEATURES}")

In [ ]:
# Feature selection summary (correlation + VIF)
selector.summary()

In [ ]:
# Correlation heatmap for selected features
fig, ax = plt.subplots(figsize=(8, 6))
corr_matrix = df_dev[SELECTED_FEATURES].corr(method="spearman")
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    ax=ax,
    linewidths=0.5,
)
ax.set_title("Spearman Correlation — Selected Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 4. 🤖 Model Training — LightGBM Regressor

We train a LightGBM regressor using the selected features on the development set.

In [ ]:
X_dev = df_dev[SELECTED_FEATURES]
y_dev = df_dev[TARGET]
X_oot = df_oot[SELECTED_FEATURES]
y_oot = df_oot[TARGET]

# Train/validation split within dev for early stopping
X_train, X_val, y_train, y_val = train_test_split(X_dev, y_dev, test_size=0.20, random_state=42)

model = lgb.LGBMRegressor(
    objective="regression",
    metric="rmse",
    n_estimators=500,
    learning_rate=0.05,
    num_leaves=63,
    feature_fraction=0.8,
    bagging_fraction=0.8,
    bagging_freq=5,
    random_state=42,
    n_jobs=-1,
    verbose=-1,
)

model.fit(
    X_train,
    y_train,
    eval_set=[(X_val, y_val)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(period=0)],
)

print(f"Best iteration : {model.best_iteration_}")
print(f"Best val score : {model.best_score_['valid_0']['rmse']:.4f} RMSE")

In [ ]:
# Generate predictions for dev and OOT sets
df_dev["prediction"] = model.predict(X_dev)
df_oot["prediction"] = model.predict(X_oot)

print("Predictions generated for dev and OOT sets ✅")
print(
    f"Dev  prediction range : [{df_dev['prediction'].min():.2f}, {df_dev['prediction'].max():.2f}]"
)
print(
    f"OOT  prediction range : [{df_oot['prediction'].min():.2f}, {df_oot['prediction'].max():.2f}]"
)

---
## 5. 📊 Model Evaluation — `RegressionEvaluator`

`RegressionEvaluator` computes standard regression metrics and produces diagnostic plots:

| Metric | Description |
|--------|-------------|
| **RMSE** | Root Mean Squared Error — penalizes large errors |
| **MAE** | Mean Absolute Error — robust to outliers |
| **R²** | Coefficient of Determination — proportion of variance explained |
| **MAPE** | Mean Absolute Percentage Error — scale-independent |

We evaluate on both the **dev** and **OOT** sets to detect overfitting.

In [ ]:
evaluator = RegressionEvaluator()

# Evaluate on dev and OOT
metrics_dev = evaluator.evaluate(y_true=df_dev[TARGET], y_pred=df_dev["prediction"])
metrics_oot = evaluator.evaluate(y_true=df_oot[TARGET], y_pred=df_oot["prediction"])

# Display side-by-side
metrics_df = pd.DataFrame({"Dev": metrics_dev, "OOT": metrics_oot})
print("📈 Regression Metrics:")
metrics_df

In [ ]:
# Residual plot on OOT set
fig, ax = plt.subplots(figsize=(8, 5))
evaluator.residual_plot(y_true=df_oot[TARGET], y_pred=df_oot["prediction"], ax=ax)
ax.set_title("Residual Plot — OOT Set", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Prediction interval coverage (simulating a +/- 0.5 interval for demo)
coverage = evaluator.prediction_interval_coverage(
    y_true=df_oot[TARGET], y_lower=df_oot["prediction"] - 0.5, y_upper=df_oot["prediction"] + 0.5
)
print(f"Prediction Interval Coverage (±0.5): {coverage:.2%}")

---
## 6. 📡 Stability Monitoring

We use `PSICalculator` to monitor **input feature drift** and `RegressionPSI` to monitor **predicted score drift** between dev and OOT.

`StabilityReport` orchestrates both checks and returns a unified result:

| PSI Value | Signal |
|-----------|--------|
| < 0.10 | 🟢 Stable — no action needed |
| 0.10 – 0.25 | 🟡 Moderate shift — monitor closely |
| > 0.25 | 🔴 Significant drift — investigate |

### 6.1 Feature PSI (input drift)

In [ ]:
# Fit PSICalculator on dev set (reference)
feature_psi = PSICalculator(n_bins=10)
feature_psi.fit(df_dev, features=SELECTED_FEATURES)

# Compute PSI on OOT set (current)
feature_psi_report = feature_psi.transform(df_oot)
print("📊 Feature PSI (Dev → OOT):")
feature_psi_report.sort_values("psi", ascending=False)

### 6.2 Score PSI — `RegressionPSI` (predicted value drift)

In [ ]:
# Fit RegressionPSI on dev predictions (reference distribution)
regression_psi = RegressionPSI(n_bins=10)
regression_psi.fit(df_dev, score_col="prediction")

# Compute score PSI on OOT
score_psi_report = regression_psi.transform(df_oot)
score_psi_value = regression_psi.get_psi()

print(f"🎯 Score PSI (Dev → OOT): {score_psi_value:.4f}")
score_psi_report

### 6.3 Full Stability Report

In [ ]:
# Build a ProjectContext for regression
ctx = ProjectContext()
ctx.task_type = TaskType.REGRESSION
ctx.target = TARGET
ctx.selected_features = SELECTED_FEATURES
ctx.training_date = datetime.now().isoformat()

# Store reference stats in context (for future monitoring runs)
ctx.reference_stats = {}
ctx.reference_stats.update(feature_psi.reference_stats_)
ctx.reference_stats.update(regression_psi.reference_stats_)

# Run unified StabilityReport using context
stability = StabilityReport(context=ctx)
full_report = stability.run(
    df=df_oot,
    features=SELECTED_FEATURES,
    score_col="prediction",
)

print("📋 Full Stability Report:")
full_report

In [ ]:
# Visualize drift using library heatmap
fig, ax = plt.subplots(figsize=(8, 6))
stability.plot_drift_heatmap(ax=ax)
plt.tight_layout()
plt.show()

---
## 7. 💾 Context Persistence — Save & Reload

`ProjectContext` serializes all metadata (selected features, reference stats, task type, training date) to disk using `joblib`. This allows you to reload the context in a future monitoring run without re-training.

**Workflow:**
1. **Training time**: fit everything, `ctx.save(path)`.
2. **Monitoring time**: `ctx = ProjectContext.load(path)`, run `StabilityReport(context=ctx).run(df_new, ...)`.

In [ ]:
import os

# Save context to disk
ctx_path = "regression_context.joblib"
ctx.save(ctx_path)
print(f"Context saved to '{ctx_path}' ✅")
print(f"File size: {os.path.getsize(ctx_path):,} bytes")

In [ ]:
# Reload context and run stability monitoring as in production
ctx_loaded = ProjectContext.load(ctx_path)

print("Loaded context summary:")
print(f"  task_type         : {ctx_loaded.task_type}")
print(f"  target            : {ctx_loaded.target}")
print(f"  selected_features : {ctx_loaded.selected_features}")
print(f"  training_date     : {ctx_loaded.training_date}")
print(f"  reference_stats   : {list(ctx_loaded.reference_stats.keys())}")

# Re-run stability report from loaded context (production scenario)
stability_prod = StabilityReport(context=ctx_loaded)
prod_report = stability_prod.run(
    df=df_oot,
    features=ctx_loaded.selected_features,
    score_col="prediction",
)

print("\n📋 Production Stability Report (from reloaded context):")
prod_report

---
## ✅ Summary

This notebook demonstrated the **complete regression lifecycle** with `model_track`:

| Step | Component | Key Output |
|------|-----------|------------|
| 1. Data Setup | sklearn / pandas | Train/OOT split |
| 2. Auditing | `DataAuditor`, `TypeDetector` | Feature type map |
| 3. Feature Selection | `RegressionSelector` (Spearman + VIF) | `selected_features_` |
| 4. Model Training | `LGBMRegressor` | Predictions |
| 5. Evaluation | `RegressionEvaluator` | RMSE, MAE, R², MAPE, residual plot |
| 6. Stability | `RegressionPSI`, `StabilityReport` | PSI per feature + score |
| 7. Persistence | `ProjectContext.save/load` | Reusable monitoring context |

### Key Takeaways

- **`RegressionSelector`** handles both correlation-based filtering and VIF-based multicollinearity removal in a single `fit()` call.
- **`RegressionPSI`** provides semantic clarity for continuous score monitoring — distinguishing regression scores from binary/multiclass probabilities.
- **`StabilityReport`** auto-detects the task type via `ProjectContext` and routes to the appropriate PSI calculator.
- **`ProjectContext`** persists all artifacts needed for production monitoring without re-training.